### Load libraries

In [ ]:
from mstr_robotics._paths import REPO_ROOT, USER_CONFIG, OSI_FILES, OSI_SCHEMA, OSI_DASHBOARD_CONTEXT, MCP_DATA, PYTHON_IO
#from flashtext import KeywordProcessor
from mstr_robotics.navigation import AnswerPrompts, MstrObjects
from mstr_robotics.report import Rep, Prompts
from mstr_robotics.mstr_classes import get_conn
from IPython.display import HTML
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path
import os
import json 

from mstr_robotics.user_rag import KeywordProcessor, Perplexity

i_prompts=Prompts()
i_rep=Rep()
u_perplexity=Perplexity()
i_mstr_objects=MstrObjects()

u_keyword_processor=KeywordProcessor()
os_mcp_folder_str=str(MCP_DATA)

user_path="..\\config\\user_d.json"
with open(user_path, 'r') as file:
    user_d = json.load(file)

env_file="..\\config\\API_KEY.env"
load_dotenv(env_file)


project_id = user_d["conn_params"]["project_id"]
conn_params =  user_d["conn_params"]
conn_params["project_id"]=project_id
conn = get_conn(**conn_params)
conn.headers['Content-type'] = "application/json"

In [ ]:
#object ids are maintained in ..\config\jupyter_objects_d.json
with open("..\\config\\jupyter_objects_d.json", "r") as openfile:
    jupyter_objects_d = json.load(openfile)
nb_d = jupyter_objects_d["jup_Colab_perplex"]

ai_rep_name="dyn_prompt_page_botstat"
ai_rep_folder_id=nb_d["folders"]["ai_rep_folder_id"]
report_id=nb_d["reports"]["report_id"]

In [ ]:
msg_t="Please show me the Cost_1, Revenue and Profit for the attributes Year,Category, Region"
msg_t= msg_t + " and filter for the yaer 2021, 2022 and 2023"
msg_t= msg_t + " and Categories starting with B"
msg_t= msg_t + " and Revenue is between 10 and 1000000"


In [ ]:
# MSTR specitic objects and definitions
attribute_form_elements_df = pd.read_csv(os.path.join(os_mcp_folder_str, "attribute_form_elements.csv"))
attribute_elements_df = pd.read_csv(os.path.join(os_mcp_folder_str, "attribute_elements.csv"))
att_form_def_df = pd.read_csv(os.path.join(os_mcp_folder_str, "att_form_def.csv"))
obj_prp_rel_df = pd.read_csv(os.path.join(os_mcp_folder_str, "obj_prp_rel.csv"))
dos_rep_prp_rel_df = pd.read_csv(os.path.join(os_mcp_folder_str, "dos_rep_prp_rel.csv"))
dashboard_definitions_df = pd.read_csv(os.path.join(os_mcp_folder_str, "dashboard_definitions.csv"))
dashboard_chapter_filter_df = pd.read_csv(os.path.join(os_mcp_folder_str, "dashboard_chapter_filter.csv"))
dashboard_selector_filter_df = pd.read_csv(os.path.join(os_mcp_folder_str, "dashboard_selector_filter.csv"))

i_answer_prompts=AnswerPrompts(attribute_form_elements_df=attribute_form_elements_df
                                , attribute_elements_df=attribute_elements_df
                                , obj_prp_rel_df=obj_prp_rel_df
                                , att_form_def_df=att_form_def_df
                                , dos_rep_prp_rel_df=dos_rep_prp_rel_df
                                , dashboard_definitions_df=dashboard_definitions_df
                                , dashboard_chapter_filter_df=dashboard_chapter_filter_df
                                , dashboard_selector_filter_df=dashboard_selector_filter_df
                                )

In [ ]:
# tool agnostic definitions. Only MSTR for the moment
element_df_d_l=[{"df":attribute_form_elements_df,"key_col":"element_val","key_type":"element_val","rag_cols": ["attribute_name", "form_name", "element_val"]},
                 {"df":attribute_elements_df,"key_col":"element_val","key_type":"element_val","rag_cols": ["attribute_name", "element_val"]}
                ]

bi_obj_df=obj_prp_rel_df[["object_name", "obj_type", "object_id"]][obj_prp_rel_df["obj_type"].isin(["attribute","metric"]) ]

obj_df_d_l=[{"df":bi_obj_df,"key_col":"object_name","key_type":"object_name","rag_cols": ["object_name", "obj_type"]}]
            


In [ ]:
key_word_l=u_keyword_processor.extract_keywords(msg_t=msg_t)

att_elem_str=i_mstr_objects.get_att_elem_str(element_df_d_l, key_word_l=key_word_l)
bi_obj_str=i_mstr_objects.get_att_elem_str(obj_df_d_l, key_word_l=key_word_l)

sys_cont=u_perplexity.rag_sys_cont(key_word_l=key_word_l,att_elem_str=att_elem_str, bi_obj_str=bi_obj_str)

message_check_d={}
message_check_d_l=[]
message_check_d["msg_nr"] = "1"
message_check_d["msg_t"] = msg_t
message_check_d=u_perplexity.call_perplexity( msg_t=msg_t, sys_cont=sys_cont, message_check_d=message_check_d, temperature=0.1)
message_check_d_l.append(message_check_d.copy())

bi_request_d=u_perplexity.parse_and_structure(message_check_d_l)
bi_request_d

In [ ]:
prompt_answ=i_answer_prompts.AI_mstr_prp_page_ans( vector_store=u_keyword_processor
                                                  ,bi_request_d=bi_request_d
                                                  ,rep_dos_id=report_id
                                                  )
prompt_answ


In [ ]:
rep_id=i_answer_prompts.save_AI_rep(conn=conn,report_id=report_id
                                  ,prompt_answ=prompt_answ
                                  ,ai_rep_name=ai_rep_name
                                  ,ai_rep_folder_id=ai_rep_folder_id)
new_rep_id=rep_id.json()["id"]
instance_id = i_rep.open_Instance(conn=conn, report_id=new_rep_id)
df=i_rep.report_df(conn=conn, report_id=new_rep_id, instance_id=instance_id)
df

In [ ]:
rep_id=i_answer_prompts.save_AI_rep(conn=conn,report_id=report_id
                                  ,prompt_answ=prompt_answ
                                  ,ai_rep_name=ai_rep_name
                                  ,promptOption ="filterAndTemplate"
                                  ,ai_rep_folder_id=ai_rep_folder_id)
new_rep_id=rep_id.json()["id"]
link=i_rep.web_base_url(conn=conn,report_id=new_rep_id)
HTML(link)